# NeuralSpace Google Colab Bootstrap

Run the single bootstrap cell below, paste your one-time claim code, then use `ns.datasets` and `ns.models` to list, refresh, or download workspace assets.

In [ ]:
from getpass import getpass
from pathlib import Path
from urllib.parse import urlparse
import socket
import requests


DEFAULT_API_BASE = "https://uniform-ate-seo-statistical.trycloudflare.com/api/v1"

class NeuralSpaceAssetClient:
    def __init__(self, parent, kind):
        self.parent = parent
        self.kind = kind

    @property
    def items(self):
        return self.parent.assets.get(self.kind, [])

    def list(self, refresh=False):
        if refresh:
            self.parent.refresh_assets()
        return self.items

    def get(self, asset_id):
        key = "dataset_id" if self.kind == "datasets" else "model_id"
        for item in self.items:
            if item.get(key) == asset_id:
                return item
        raise KeyError(f"{self.kind[:-1]} not attached to this workspace: {asset_id}")

    def download(self, asset_id, target_dir=None):
        item = self.get(asset_id)
        signed_url = item.get("signed_url")
        if not signed_url:
            raise RuntimeError(f"No download URL available for {asset_id}")
        base_dir = Path(target_dir or f"/content/neuralspace/{self.kind}/{asset_id}")
        base_dir.mkdir(parents=True, exist_ok=True)
        filename = Path(urlparse(signed_url).path).name or f"{asset_id}.bin"
        output_path = base_dir / filename
        with requests.get(signed_url, stream=True, timeout=120) as response:
            response.raise_for_status()
            with output_path.open("wb") as handle:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        handle.write(chunk)
        return str(output_path)

    def load(self, asset_id, target_dir=None):
        return self.download(asset_id, target_dir=target_dir)


class NeuralSpaceClient:
    def __init__(self, api_base, runtime_token, bootstrap):
        self.api_base = api_base.rstrip("/")
        self.runtime_token = runtime_token
        self.headers = {"Authorization": f"Bearer {runtime_token}"}
        self.session_id = bootstrap["session_id"]
        self.assets = {
            "datasets": bootstrap.get("datasets", []),
            "models": bootstrap.get("models", []),
        }
        self.datasets = NeuralSpaceAssetClient(self, "datasets")
        self.models = NeuralSpaceAssetClient(self, "models")

    @classmethod
    def connect(cls, api_base=None):
        api_base = (api_base or DEFAULT_API_BASE).strip().rstrip("/")
        print(f"Using NeuralSpace API: {api_base}")
        if not (api_base.startswith("https://") or api_base.startswith("http://localhost") or api_base.startswith("http://127.0.0.1")):
            raise RuntimeError("Use HTTPS for remote NeuralSpace APIs")
        host = urlparse(api_base).hostname
        try:
            socket.gethostbyname(host)
        except OSError as exc:
            raise RuntimeError(
                f"Cannot resolve NeuralSpace API host: {host}. "
                "If you are using trycloudflare.com, restart the Cloudflare tunnel and paste the fresh public URL."
            ) from exc
        claim_code = getpass("Paste NeuralSpace claim code: ").strip()
        response = requests.post(f"{api_base}/colab/claims/exchange", json={"claim_code": claim_code}, timeout=30)
        response.raise_for_status()
        return cls(api_base, response.json()["runtime_token"], response.json())

    def request(self, method, path, **kwargs):
        response = requests.request(method, f"{self.api_base}{path}", headers=self.headers, timeout=kwargs.pop("timeout", 30), **kwargs)
        response.raise_for_status()
        return response.json()

    def heartbeat(self):
        return self.request("POST", "/colab/runtime/heartbeat")

    def refresh_assets(self):
        self.assets = self.request("GET", "/colab/runtime/assets")
        return self.assets

    def start_run(self, name="Colab run"):
        return self.request("POST", "/colab/runtime/runs", json={"name": name})


ns = NeuralSpaceClient.connect()
assets = ns.refresh_assets()
datasets = ns.datasets.list()
models = ns.models.list()
print(f"Connected to NeuralSpace session {ns.session_id}")
print(f"Datasets: {len(datasets)} | Models: {len(models)}")
assets

In [ ]:
# Optional smoke test: send a heartbeat and create a run.
ns.heartbeat()
run = ns.start_run("Colab workspace smoke test")
print("Run created:", run["run_id"])